<a href="https://colab.research.google.com/github/kimkihyun1/ssafy-ai-challenge-vqa/blob/main/vqa_qwen35_27b_LASTMILE_BASEOFF_MULTICROP_FINAL.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# SSAFY VQA — Qwen3.5-27B Last-Mile 점수 개선

현재 최고 제출을 **다시 학습하지 않고** 마지막으로 보정하는 inference-only 노트북입니다.

현재까지의 기준:
- Qwen3.5-27B + BF16 LoRA
- 전체 train 5,073개 / 1 epoch
- visual 1024
- 기존 TTA4: Kaggle **0.95072**
- Adaptive TTA24 + HiRes1536 W35도 **0.95072**

따라서 이번에는 같은 fine-tuned 모델을 반복하는 대신 서로 다른 증거를 추가합니다.

```text
현재 best 확률
(TTA24 + HiRes1536 W35)
        │
        ├── LoRA OFF: 원본 Qwen3.5-27B TTA4
        │
        └── fine-tuned LoRA ON: multi-crop TTA4
                      │
                      ↓
         보수적인 selective correction
                      ↓
              여러 submission 생성
```

핵심 원칙은 **현재 95% 모델의 확신이 높은 답은 건드리지 않고,
불확실한 소수 샘플만 다른 시각적 증거가 강하게 동의할 때 수정**하는 것입니다.

기본 추가 추론량:
- LoRA OFF: 어려운 500개 × TTA4 = 2,000회
- Multi-crop: 어려운 200개 × 5 crops × TTA4 = 4,000회
- 총 약 6,000회

기존 20,296회 TTA4보다 훨씬 작습니다.

## 0. 패키지 설치

새 Colab 런타임이면 한 번 실행 후 **런타임 → 세션 다시 시작**하세요.
재시작 후에는 Drive mount부터 실행합니다.

In [ ]:
!pip uninstall -y torchao
!pip install -q -U git+https://github.com/huggingface/transformers
!pip install -q -U \
    "peft>=0.20.0" \
    "accelerate>=1.8.0" \
    "bitsandbytes>=0.46.1" \
    "pandas==2.2.3" \
    pillow tqdm safetensors

print("설치 완료 ✅")
print('처음 설치했다면 "런타임 > 세션 다시 시작" 후 Drive mount부터 실행하세요.')

Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 104.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.3/57.3 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 65.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 147.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.2/80.2 kB 8.8 MB/s eta 0:00:00
설치 완료 ✅
처음 설치했다면 "런타임 > 세션 다시 시작" 후 Drive mount부터 실행하세요.


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


## 1. 설정

In [ ]:
ROOT = "/content/drive/MyDrive/ssafy-16-1-ai-8-28"

MODEL_ID = "Qwen/Qwen3.5-27B"

FINAL_ADAPTER_DIR = (
    "/content/drive/MyDrive/ssafy-16-1-ai-8-28/"
    "qwen35_27b_FINAL_full5073_bf16_v1024_epoch1"
)

RUN_CONFIG_PATH = (
    "/content/drive/MyDrive/ssafy-16-1-ai-8-28/"
    "qwen35_27b_FINAL_full5073_v1024_epoch1_run_config.json"
)

BASE_RUN_TAG = "qwen35_27b_full5073_bf16_v1024_e1"

BASE_TTA_CACHE_DIR = (
    "/content/drive/MyDrive/ssafy-16-1-ai-8-28/"
    "tta_cache_qwen35_27b_full5073_bf16_v1024_e1"
)

# 이전 Adaptive notebook과 같은 값
ADAPTIVE_HARD_COUNT = 1000
ADAPTIVE_EXTRA_PERMS = 20
HIRES_COUNT = 400
HIRES_VISUAL_TOKENS = 1536
CURRENT_BEST_HIRES_WEIGHT = 0.35

# 이번 추가 계산량
BASE_OFF_COUNT = 500
MULTICROP_COUNT = 200

# 5 crop × TTA4
MULTICROP_TTA_PERMS = 4
CROP_FRACTION = 0.72

COPY_TEST_TO_LOCAL = True
INFER_BATCH = 1
NUM_WORKERS = 0

CHOICES = ["a", "b", "c", "d"]

print("BASE_OFF_COUNT:", BASE_OFF_COUNT)
print("MULTICROP_COUNT:", MULTICROP_COUNT)
print("MULTICROP_TTA_PERMS:", MULTICROP_TTA_PERMS)
print("CROP_FRACTION:", CROP_FRACTION)

BASE_OFF_COUNT: 500
MULTICROP_COUNT: 200
MULTICROP_TTA_PERMS: 4
CROP_FRACTION: 0.72


## 2. 환경 / test 데이터

In [ ]:
import os
import gc
import json
import shutil
import hashlib
import itertools
from pathlib import Path
from dataclasses import dataclass
from typing import Any
from contextlib import nullcontext

import numpy as np
import pandas as pd
from PIL import Image
from tqdm.auto import tqdm

import torch
from torch.utils.data import Dataset, DataLoader

import transformers
import peft

from transformers import (
    AutoProcessor,
    AutoModelForMultimodalLM,
    BitsAndBytesConfig,
)

from peft import PeftModel

Image.MAX_IMAGE_PIXELS = None
os.environ["TOKENIZERS_PARALLELISM"] = "false"

assert torch.cuda.is_available(), "A100 GPU 런타임으로 변경하세요."

GPU_NAME = torch.cuda.get_device_name(0)
GPU_GB = torch.cuda.get_device_properties(0).total_memory / 1024**3
DTYPE = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16

print("PyTorch     :", torch.__version__)
print("Transformers:", transformers.__version__)
print("PEFT        :", peft.__version__)
print("GPU         :", GPU_NAME)
print(f"VRAM        : {GPU_GB:.1f} GB")
print("dtype       :", DTYPE)

ROOT = Path(ROOT)
FINAL_ADAPTER_DIR = Path(FINAL_ADAPTER_DIR)
RUN_CONFIG_PATH = Path(RUN_CONFIG_PATH)
BASE_TTA_CACHE_DIR = Path(BASE_TTA_CACHE_DIR)

TEST_CSV = ROOT / "test.csv"
SAMPLE_CSV = ROOT / "sample_submission.csv"

test_df = pd.read_csv(TEST_CSV)
sample_df = pd.read_csv(SAMPLE_CSV) if SAMPLE_CSV.exists() else None

test_df = test_df.drop_duplicates("id").reset_index(drop=True)

required_test = {"id", "path", "question", "a", "b", "c", "d"}
assert required_test.issubset(test_df.columns), test_df.columns

assert FINAL_ADAPTER_DIR.exists(), f"adapter not found: {FINAL_ADAPTER_DIR}"
assert BASE_TTA_CACHE_DIR.exists(), f"TTA4 cache not found: {BASE_TTA_CACHE_DIR}"

SUBMISSION_DIR = ROOT / "submission"
SUBMISSION_DIR.mkdir(parents=True, exist_ok=True)

LASTMILE_DIR = ROOT / "lastmile_qwen35_27b"
LASTMILE_DIR.mkdir(parents=True, exist_ok=True)

print("test:", test_df.shape)
print("adapter exists ✅")
print("base TTA4 cache exists ✅")

PyTorch     : 2.11.0+cu128
Transformers: 5.16.0.dev0
PEFT        : 0.20.0
GPU         : NVIDIA A100-SXM4-80GB
VRAM        : 79.3 GB
dtype       : torch.bfloat16
test: (5074, 7)
adapter exists ✅
base TTA4 cache exists ✅


## 3. test 이미지를 로컬 SSD로 복사

In [ ]:
LOCAL_TEST_DIR = Path("/content/vqa_data/test")
DRIVE_TEST_DIR = ROOT / "test"

if COPY_TEST_TO_LOCAL:
    if not LOCAL_TEST_DIR.exists():
        LOCAL_TEST_DIR.parent.mkdir(parents=True, exist_ok=True)
        print(f"copy: {DRIVE_TEST_DIR} -> {LOCAL_TEST_DIR}")
        shutil.copytree(DRIVE_TEST_DIR, LOCAL_TEST_DIR)
    else:
        print("already exists:", LOCAL_TEST_DIR)

def resolve_test_image_path(p):
    p = Path(str(p))

    candidates = []

    if p.is_absolute():
        candidates.append(p)
    else:
        candidates.extend([
            LOCAL_TEST_DIR / p.name,
            ROOT / p,
            ROOT / "test" / p.name,
        ])

    for c in candidates:
        if c.exists():
            return c

    raise FileNotFoundError(f"test image not found: {p}")

print("first test image:", resolve_test_image_path(test_df.iloc[0]["path"]))

copy: /content/drive/MyDrive/ssafy-16-1-ai-8-28/test -> /content/vqa_data/test
first test image: /content/vqa_data/test/test_0001.jpg


## 4. 기존 TTA4 + Adaptive TTA24 + HiRes1536 cache 복원

이 단계는 GPU를 사용하지 않습니다.

이전 노트북과 **동일한 불확실도 계산/샘플 선택**을 재현해서
이미 저장된 Adaptive TTA24와 HiRes1536 cache를 찾아 현재 best 확률을 복원합니다.

In [ ]:
BASE_PERMS = [
    (0, 1, 2, 3),
    (1, 3, 0, 2),
    (2, 0, 3, 1),
    (3, 2, 1, 0),
]

ALL_PERMS = list(itertools.permutations(range(4)))
REMAINING_PERMS = [p for p in ALL_PERMS if p not in BASE_PERMS]

assert len(REMAINING_PERMS) == 20

def uncertainty_metrics(runs):
    mean_probs = runs.mean(axis=0)

    sorted_probs = np.sort(mean_probs, axis=1)
    margin = sorted_probs[:, -1] - sorted_probs[:, -2]

    confidence = mean_probs.max(axis=1)

    entropy = -np.sum(
        mean_probs * np.log(np.clip(mean_probs, 1e-12, 1.0)),
        axis=1,
    ) / np.log(4.0)

    votes = runs.argmax(axis=2)

    vote_agreement = np.zeros(runs.shape[1], dtype=np.float64)

    for i in range(runs.shape[1]):
        counts = np.bincount(votes[:, i], minlength=4)
        vote_agreement[i] = counts.max() / runs.shape[0]

    score = (
        0.40 * (1.0 - margin)
        + 0.30 * entropy
        + 0.15 * (1.0 - confidence)
        + 0.15 * (1.0 - vote_agreement)
    )

    return {
        "mean_probs": mean_probs,
        "margin": margin,
        "confidence": confidence,
        "entropy": entropy,
        "vote_agreement": vote_agreement,
        "score": score,
        "votes": votes,
    }

# ------------------------------------------------------------
# 기존 TTA4
# ------------------------------------------------------------
base_runs = []
ids_ref = None

for pi in range(1, 5):
    f = BASE_TTA_CACHE_DIR / f"{BASE_RUN_TAG}_perm{pi}.npz"
    assert f.exists(), f"missing base cache: {f}"

    z = np.load(f, allow_pickle=True)

    probs = z["original_probs"].astype(np.float64)
    ids = [str(x) for x in z["ids"].tolist()]

    if ids_ref is None:
        ids_ref = ids
    else:
        assert ids == ids_ref

    base_runs.append(probs)

base_runs = np.stack(base_runs, axis=0)

assert base_runs.shape == (4, len(test_df), 4)
assert ids_ref == test_df["id"].astype(str).tolist()

base_mean = base_runs.mean(axis=0)
base_unc = uncertainty_metrics(base_runs)

# ------------------------------------------------------------
# 이전 Adaptive hard 1000 재현
# ------------------------------------------------------------
hard_indices = np.argsort(
    -base_unc["score"]
)[:min(ADAPTIVE_HARD_COUNT, len(test_df))]

hard_indices = np.sort(hard_indices)

hard_ids = test_df.iloc[hard_indices]["id"].astype(str).tolist()

hard_hash = hashlib.sha1(
    "\n".join(hard_ids).encode("utf-8")
).hexdigest()[:10]

hard_df = test_df.iloc[hard_indices].copy().reset_index(drop=True)

EXTRA_CACHE_DIR = (
    ROOT
    / "adaptive_tta27b"
    / (
        f"extra_v1024_h{len(hard_df)}_"
        f"{hard_hash}_k{ADAPTIVE_EXTRA_PERMS}"
    )
)

assert EXTRA_CACHE_DIR.exists(), (
    "Adaptive TTA24 cache를 찾지 못했습니다. "
    f"예상 경로: {EXTRA_CACHE_DIR}"
)

extra_runs = []

for ei, perm in enumerate(REMAINING_PERMS, start=1):
    perm_key = "".join(map(str, perm))
    f = EXTRA_CACHE_DIR / f"extra_{ei:02d}_perm{perm_key}.npz"

    assert f.exists(), f"missing adaptive cache: {f}"

    z = np.load(f, allow_pickle=True)

    probs = z["original_probs"].astype(np.float64)
    ids = [str(x) for x in z["ids"].tolist()]

    assert ids == hard_df["id"].astype(str).tolist()

    extra_runs.append(probs)

extra_runs = np.stack(extra_runs, axis=0)

hard_base_runs = base_runs[:, hard_indices, :]

hard_all_runs = np.concatenate(
    [hard_base_runs, extra_runs],
    axis=0,
)

assert hard_all_runs.shape[0] == 24

hard_arith = hard_all_runs.mean(axis=0)

adaptive_arith = base_mean.copy()
adaptive_arith[hard_indices] = hard_arith

# ------------------------------------------------------------
# 이전 HiRes 400 재현
# ------------------------------------------------------------
hard_unc_after = uncertainty_metrics(hard_all_runs)

hires_local_rank = np.argsort(
    -hard_unc_after["score"]
)[:min(HIRES_COUNT, len(hard_df))]

hires_global_indices = hard_indices[hires_local_rank]
hires_global_indices = np.sort(hires_global_indices)

hires_df = (
    test_df
    .iloc[hires_global_indices]
    .copy()
    .reset_index(drop=True)
)

hires_ids = hires_df["id"].astype(str).tolist()

hires_hash = hashlib.sha1(
    "\n".join(hires_ids).encode("utf-8")
).hexdigest()[:10]

HIRES_CACHE_DIR = (
    ROOT
    / "adaptive_tta27b"
    / (
        f"hires_v{HIRES_VISUAL_TOKENS}_"
        f"h{len(hires_df)}_{hires_hash}_tta4"
    )
)

assert HIRES_CACHE_DIR.exists(), (
    "HiRes1536 cache를 찾지 못했습니다. "
    f"예상 경로: {HIRES_CACHE_DIR}"
)

hires_runs = []

for pi, perm in enumerate(BASE_PERMS, start=1):
    perm_key = "".join(map(str, perm))
    f = HIRES_CACHE_DIR / f"hires_perm{pi}_{perm_key}.npz"

    assert f.exists(), f"missing hires cache: {f}"

    z = np.load(f, allow_pickle=True)

    probs = z["original_probs"].astype(np.float64)
    ids = [str(x) for x in z["ids"].tolist()]

    assert ids == hires_df["id"].astype(str).tolist()

    hires_runs.append(probs)

hires_runs = np.stack(hires_runs, axis=0)
hires_mean = hires_runs.mean(axis=0)
hires_unc = uncertainty_metrics(hires_runs)

# ------------------------------------------------------------
# 사용자 현재 best = Adaptive arithmetic + HiRes W35
# ------------------------------------------------------------
current_best_probs = adaptive_arith.copy()

current_best_probs[hires_global_indices] = (
    (1.0 - CURRENT_BEST_HIRES_WEIGHT)
    * adaptive_arith[hires_global_indices]
    +
    CURRENT_BEST_HIRES_WEIGHT
    * hires_mean
)

current_best_pred = current_best_probs.argmax(axis=1)

sorted_current = np.sort(current_best_probs, axis=1)
current_margin = sorted_current[:, -1] - sorted_current[:, -2]
current_conf = current_best_probs.max(axis=1)

print("base TTA4:", base_runs.shape)
print("adaptive hard:", len(hard_df))
print("adaptive runs:", hard_all_runs.shape)
print("hires:", hires_runs.shape)
print("current best reconstructed ✅")

base TTA4: (4, 5074, 4)
adaptive hard: 1000
adaptive runs: (24, 1000, 4)
hires: (4, 400, 4)
current best reconstructed ✅


## 5. GPU 0원 후보 — 기존 HiRes weight 45%, 50%

W35가 지금 최고였으므로 그 근처의 W45/W50은 **추가 추론 없이** 만들 수 있습니다.

이 후보들은 먼저 만들어지며 A100 계산을 사용하지 않습니다.

In [ ]:
def save_submission_from_probs(probs, filename):
    pred = np.array(CHOICES)[probs.argmax(axis=1)]

    sub = pd.DataFrame({
        "id": test_df["id"].values,
        "answer": pred,
    })

    if sample_df is not None:
        assert set(sub["id"]) == set(sample_df["id"])

        sub = sample_df[["id"]].merge(
            sub,
            on="id",
            how="left",
        )

    assert sub["answer"].notna().all()
    assert set(sub["answer"].unique()).issubset(set(CHOICES))

    out = SUBMISSION_DIR / filename
    sub.to_csv(out, index=False)

    return out, sub

FREE_OUTPUTS = []

for w in [0.45, 0.50]:
    p = adaptive_arith.copy()

    p[hires_global_indices] = (
        (1.0 - w)
        * adaptive_arith[hires_global_indices]
        +
        w
        * hires_mean
    )

    out, sub = save_submission_from_probs(
        p,
        f"submission_qwen35_27b_ADAPTIVE_TTA24_H1000_HI1536_N400_W{int(w*100)}.csv",
    )

    changed = int(
        (p.argmax(axis=1) != current_best_pred).sum()
    )

    FREE_OUTPUTS.append((w, out, changed))

print("무료 후보:")
for w, out, changed in FREE_OUTPUTS:
    print(f"W{int(w*100)} | current W35 대비 changed={changed} | {out}")

무료 후보:
W45 | current W35 대비 changed=2 | /content/drive/MyDrive/ssafy-16-1-ai-8-28/submission/submission_qwen35_27b_ADAPTIVE_TTA24_H1000_HI1536_N400_W45.csv
W50 | current W35 대비 changed=2 | /content/drive/MyDrive/ssafy-16-1-ai-8-28/submission/submission_qwen35_27b_ADAPTIVE_TTA24_H1000_HI1536_N400_W50.csv


## 6. Processor / prompt / 일반 inference 함수

In [ ]:
VISUAL_TOKEN_SIDE = 32
MIN_VISUAL_TOKENS = 256
BASE_VISUAL_TOKENS = 1024

MIN_PIXELS = (
    MIN_VISUAL_TOKENS
    * VISUAL_TOKEN_SIDE
    * VISUAL_TOKEN_SIDE
)

MAX_PIXELS = (
    BASE_VISUAL_TOKENS
    * VISUAL_TOKEN_SIDE
    * VISUAL_TOKEN_SIDE
)

processor = AutoProcessor.from_pretrained(
    MODEL_ID,
    min_pixels=MIN_PIXELS,
    max_pixels=MAX_PIXELS,
)

processor.tokenizer.padding_side = "right"

SYSTEM_INSTRUCT = (
    "You are a highly accurate visual multiple-choice question answering model. "
    "Use the image and the answer choices. "
    "Return exactly one lowercase letter: a, b, c, or d."
)

def build_mc_prompt(question, options):
    return (
        f"질문: {question}\n"
        f"(a) {options[0]}\n"
        f"(b) {options[1]}\n"
        f"(c) {options[2]}\n"
        f"(d) {options[3]}\n\n"
        "이미지와 선택지를 함께 판단하세요. "
        "최종 답은 a, b, c, d 중 한 글자만 출력하세요."
    )

def make_messages(image, question, options):
    return [
        {
            "role": "system",
            "content": [
                {"type": "text", "text": SYSTEM_INSTRUCT}
            ],
        },
        {
            "role": "user",
            "content": [
                {"type": "image", "image": image},
                {"type": "text", "text": build_mc_prompt(question, options)},
            ],
        },
    ]

def apply_template(messages):
    return processor.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False,
    )

choice_token_ids = []

for c in CHOICES:
    ids = processor.tokenizer.encode(
        c,
        add_special_tokens=False,
    )

    print(c, ids)

    assert len(ids) == 1, f"{c} is not single token: {ids}"
    choice_token_ids.append(ids[0])

CHOICE_TOKEN_IDS = torch.tensor(
    choice_token_ids,
    dtype=torch.long,
)

def last_nonpad_index(attention_mask):
    rev = torch.flip(attention_mask, dims=[1])
    offset = rev.float().argmax(dim=1)

    return (
        attention_mask.shape[1]
        - 1
        - offset
    )

def get_model_device(model):
    return next(model.parameters()).device

preprocessor_config.json:   0%|          | 0.00/390 [00:00<?, ?B/s]

chat_template.jinja:   0%|          | 0.00/7.76k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/4.13k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/16.7k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/6.72M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/3.35M [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 12.8MB            

tokenizer.json: downloading bytes:           |  0.00B            

video_preprocessor_config.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

a [64]
b [65]
c [66]
d [67]


In [ ]:
class EvalSubsetDataset(Dataset):
    def __init__(self, df, perm):
        self.df = df.reset_index(drop=True)
        self.perm = tuple(perm)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        image = Image.open(
            resolve_test_image_path(row["path"])
        ).convert("RGB")

        orig_options = [str(row[c]) for c in CHOICES]
        shown_options = [orig_options[j] for j in self.perm]

        return {
            "messages": make_messages(
                image,
                str(row["question"]),
                shown_options,
            ),
            "image": image,
            "id": str(row["id"]),
        }

@dataclass
class EvalSubsetCollator:
    processor: Any

    def __call__(self, batch):
        texts = [
            apply_template(x["messages"])
            for x in batch
        ]

        images = [x["image"] for x in batch]

        enc = self.processor(
            text=texts,
            images=images,
            padding=True,
            return_tensors="pt",
        )

        ids = [x["id"] for x in batch]

        return enc, ids

@torch.inference_mode()
def predict_subset_original_probs(
    model,
    df,
    perm,
    desc,
    disable_adapter=False,
):
    device = get_model_device(model)

    ds = EvalSubsetDataset(
        df,
        perm=perm,
    )

    dl = DataLoader(
        ds,
        batch_size=INFER_BATCH,
        shuffle=False,
        collate_fn=EvalSubsetCollator(processor),
        num_workers=NUM_WORKERS,
        pin_memory=True,
    )

    choice_ids = CHOICE_TOKEN_IDS.to(device)

    all_original_probs = []
    ids_all = []

    model.eval()

    if disable_adapter:
        assert hasattr(model, "disable_adapter"), (
            "현재 PEFT 버전에서 disable_adapter()를 찾지 못했습니다."
        )
        adapter_ctx = model.disable_adapter()
    else:
        adapter_ctx = nullcontext()

    with adapter_ctx:
        for enc, ids in tqdm(dl, desc=desc):
            enc = {
                k: (
                    v.to(device, non_blocking=True)
                    if torch.is_tensor(v)
                    else v
                )
                for k, v in enc.items()
            }

            with torch.autocast(
                "cuda",
                dtype=DTYPE,
            ):
                out = model(
                    **enc,
                    use_cache=False,
                )

            logits_device = out.logits.device

            idx = last_nonpad_index(
                enc["attention_mask"]
            ).to(logits_device)

            bidx = torch.arange(
                out.logits.shape[0],
                device=logits_device,
            )

            next_logits = (
                out.logits[bidx, idx]
                [:, choice_ids.to(logits_device)]
            )

            shown_probs = torch.softmax(
                next_logits.float(),
                dim=-1,
            ).cpu().numpy()

            original_probs = np.zeros_like(shown_probs)

            for shown_j, orig_j in enumerate(perm):
                original_probs[:, orig_j] = shown_probs[:, shown_j]

            all_original_probs.append(original_probs)
            ids_all.extend(ids)

            del out, next_logits, shown_probs, original_probs, enc

    return (
        np.concatenate(all_original_probs, axis=0),
        ids_all,
    )

def cached_predict(
    model,
    df,
    perm,
    cache_file,
    desc,
    disable_adapter=False,
):
    cache_file = Path(cache_file)
    expected_ids = df["id"].astype(str).tolist()

    if cache_file.exists():
        z = np.load(cache_file, allow_pickle=True)

        probs = z["original_probs"]
        ids = [str(x) for x in z["ids"].tolist()]

        assert ids == expected_ids

        print("[cache]", cache_file.name)
        return probs

    probs, ids = predict_subset_original_probs(
        model,
        df,
        perm=perm,
        desc=desc,
        disable_adapter=disable_adapter,
    )

    assert ids == expected_ids

    cache_file.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    np.savez_compressed(
        cache_file,
        original_probs=probs,
        ids=np.array(ids, dtype=object),
        perm=np.array(perm, dtype=np.int64),
    )

    print("saved:", cache_file)

    return probs

## 7. 저장된 fine-tuned LoRA 모델 로드

이 모델 하나만 VRAM에 올립니다.

LoRA OFF 추론은 같은 `PeftModel`에서 `disable_adapter()` context를 사용하므로
27B 모델을 두 개 동시에 올리지 않습니다.

In [ ]:
gc.collect()
torch.cuda.empty_cache()

precision = "bf16"

if RUN_CONFIG_PATH.exists():
    with open(RUN_CONFIG_PATH, "r", encoding="utf-8") as f:
        cfg = json.load(f)

    precision = str(cfg.get("precision", "bf16")).lower()

print("saved precision:", precision)

if precision == "bf16":
    base_model = AutoModelForMultimodalLM.from_pretrained(
        MODEL_ID,
        torch_dtype=torch.bfloat16,
        attn_implementation="sdpa",
        low_cpu_mem_usage=True,
        device_map={"": 0},
    )

elif precision == "8bit":
    quant_config = BitsAndBytesConfig(
        load_in_8bit=True,
    )

    base_model = AutoModelForMultimodalLM.from_pretrained(
        MODEL_ID,
        torch_dtype=torch.bfloat16,
        quantization_config=quant_config,
        attn_implementation="sdpa",
        low_cpu_mem_usage=True,
        device_map={"": 0},
    )

else:
    raise ValueError(precision)

final_model = PeftModel.from_pretrained(
    base_model,
    FINAL_ADAPTER_DIR,
    is_trainable=False,
)

final_model.eval()

assert hasattr(final_model, "disable_adapter")

allocated = torch.cuda.memory_allocated() / 1024**3
reserved = torch.cuda.memory_reserved() / 1024**3

print("fine-tuned adapter loaded ✅")
print("LoRA OFF context available ✅")
print(f"GPU allocated={allocated:.2f} GB, reserved={reserved:.2f} GB")

saved precision: bf16


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json:   0%|          | 0.00/127k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 11 files:   0%|          | 0/11 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/1184 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/244 [00:00<?, ?B/s]

fine-tuned adapter loaded ✅
LoRA OFF context available ✅
GPU allocated=51.40 GB, reserved=51.86 GB


# 8. 가장 어려운 500개 — LoRA OFF Qwen3.5-27B TTA4

선택은 Adaptive TTA24 이후의 불확실도를 기준으로 합니다.

원래 Qwen3.5-27B가 fine-tuning 후 모델과 다른 답을 내는지를 보는
**독립적인 tie-breaker** 역할입니다.

In [ ]:
baseoff_local_rank = np.argsort(
    -hard_unc_after["score"]
)[:min(BASE_OFF_COUNT, len(hard_df))]

baseoff_global_indices = hard_indices[baseoff_local_rank]
baseoff_global_indices = np.sort(baseoff_global_indices)

baseoff_df = (
    test_df
    .iloc[baseoff_global_indices]
    .copy()
    .reset_index(drop=True)
)

baseoff_ids = baseoff_df["id"].astype(str).tolist()

baseoff_hash = hashlib.sha1(
    "\n".join(baseoff_ids).encode("utf-8")
).hexdigest()[:10]

BASEOFF_CACHE_DIR = (
    LASTMILE_DIR
    /
    f"baseoff_v1024_n{len(baseoff_df)}_{baseoff_hash}_tta4"
)

baseoff_runs = []

for pi, perm in enumerate(BASE_PERMS, start=1):
    perm_key = "".join(map(str, perm))

    f = (
        BASEOFF_CACHE_DIR
        /
        f"baseoff_perm{pi}_{perm_key}.npz"
    )

    probs = cached_predict(
        final_model,
        baseoff_df,
        perm=perm,
        cache_file=f,
        desc=f"LoRA OFF TTA {pi}/4",
        disable_adapter=True,
    )

    baseoff_runs.append(probs)

baseoff_runs = np.stack(baseoff_runs, axis=0)
baseoff_mean = baseoff_runs.mean(axis=0)
baseoff_unc = uncertainty_metrics(baseoff_runs)

current_subset = current_best_probs[baseoff_global_indices]

cur_pred_sub = current_subset.argmax(axis=1)
baseoff_pred = baseoff_mean.argmax(axis=1)

print()
print("LoRA OFF runs:", baseoff_runs.shape)
print(
    "current best와 LoRA OFF 답이 다른 수:",
    int((cur_pred_sub != baseoff_pred).sum()),
)
print(
    "LoRA OFF TTA4 unanimous:",
    int((baseoff_unc["vote_agreement"] == 1.0).sum()),
    "/",
    len(baseoff_df),
)

LoRA OFF TTA 1/4:   0%|          | 0/500 [00:00<?, ?it/s]

saved: /content/drive/MyDrive/ssafy-16-1-ai-8-28/lastmile_qwen35_27b/baseoff_v1024_n500_bd5d629387_tta4/baseoff_perm1_0123.npz


LoRA OFF TTA 2/4:   0%|          | 0/500 [00:00<?, ?it/s]

saved: /content/drive/MyDrive/ssafy-16-1-ai-8-28/lastmile_qwen35_27b/baseoff_v1024_n500_bd5d629387_tta4/baseoff_perm2_1302.npz


LoRA OFF TTA 3/4:   0%|          | 0/500 [00:00<?, ?it/s]

saved: /content/drive/MyDrive/ssafy-16-1-ai-8-28/lastmile_qwen35_27b/baseoff_v1024_n500_bd5d629387_tta4/baseoff_perm3_2031.npz


LoRA OFF TTA 4/4:   0%|          | 0/500 [00:00<?, ?it/s]

saved: /content/drive/MyDrive/ssafy-16-1-ai-8-28/lastmile_qwen35_27b/baseoff_v1024_n500_bd5d629387_tta4/baseoff_perm4_3210.npz

LoRA OFF runs: (4, 500, 4)
current best와 LoRA OFF 답이 다른 수: 233
LoRA OFF TTA4 unanimous: 174 / 500


## 9. LoRA OFF 기반 보수적 후보 생성

세 가지를 만듭니다.

- `BASEOFF_BLEND10`: 어려운 500개에 base model 10% 확률만 추가
- `BASEOFF_STRICT`: 현재 답을 매우 제한적으로 변경
- `BASEOFF_BALANCED`: strict보다 약간 완화

Selective 후보는 **현재 margin이 낮고 base model이 강하게 확신하며,
fine-tuned TTA24에서도 base 답이 완전히 배제되지 않은 경우**만 바꿉니다.

In [ ]:
# global index -> hard local index
global_to_hard_local = {
    int(g): i
    for i, g in enumerate(hard_indices)
}

BASEOFF_OUTPUTS = []

# ------------------------------------------------------------
# A. soft blend 10%
# ------------------------------------------------------------
p_blend10 = current_best_probs.copy()

p_blend10[baseoff_global_indices] = (
    0.90 * current_best_probs[baseoff_global_indices]
    +
    0.10 * baseoff_mean
)

out_blend10, _ = save_submission_from_probs(
    p_blend10,
    "submission_qwen35_27b_LASTMILE_BASEOFF_BLEND10.csv",
)

BASEOFF_OUTPUTS.append(
    (
        "BASEOFF_BLEND10",
        out_blend10,
        int((p_blend10.argmax(axis=1) != current_best_pred).sum()),
    )
)

# ------------------------------------------------------------
# B/C. selective override
# ------------------------------------------------------------
p_strict = current_best_probs.copy()
p_balanced = current_best_probs.copy()

strict_changed = []
balanced_changed = []

for local_i, global_i in enumerate(baseoff_global_indices):
    global_i = int(global_i)

    cur_p = current_best_probs[global_i]
    cur_pred = int(cur_p.argmax())

    b_p = baseoff_mean[local_i]
    b_pred = int(b_p.argmax())

    if b_pred == cur_pred:
        continue

    hard_local = global_to_hard_local[global_i]
    ft24_p = hard_arith[hard_local]

    cur_gap = float(
        np.sort(cur_p)[-1] - np.sort(cur_p)[-2]
    )

    base_conf = float(b_p[b_pred])
    base_agree = float(baseoff_unc["vote_agreement"][local_i])

    # fine-tuned TTA24가 base 답을 얼마나 배제하는지
    support_gap = float(
        ft24_p[cur_pred] - ft24_p[b_pred]
    )

    strict_ok = (
        cur_gap < 0.10
        and base_conf >= 0.65
        and base_agree >= 1.00
        and support_gap <= 0.08
    )

    balanced_ok = (
        cur_gap < 0.15
        and base_conf >= 0.60
        and base_agree >= 0.75
        and support_gap <= 0.12
    )

    if strict_ok:
        # label을 강제로 one-hot으로 만들지 않고
        # base 증거와 현재 증거를 평균하여 자연스럽게 수정
        p_strict[global_i] = (
            0.45 * cur_p
            +
            0.55 * b_p
        )
        strict_changed.append(global_i)

    if balanced_ok:
        p_balanced[global_i] = (
            0.50 * cur_p
            +
            0.50 * b_p
        )
        balanced_changed.append(global_i)

out_strict, _ = save_submission_from_probs(
    p_strict,
    "submission_qwen35_27b_LASTMILE_BASEOFF_STRICT.csv",
)

out_balanced, _ = save_submission_from_probs(
    p_balanced,
    "submission_qwen35_27b_LASTMILE_BASEOFF_BALANCED.csv",
)

BASEOFF_OUTPUTS.extend([
    (
        "BASEOFF_STRICT",
        out_strict,
        int((p_strict.argmax(axis=1) != current_best_pred).sum()),
    ),
    (
        "BASEOFF_BALANCED",
        out_balanced,
        int((p_balanced.argmax(axis=1) != current_best_pred).sum()),
    ),
])

print("LoRA OFF 후보:")
for name, out, changed in BASEOFF_OUTPUTS:
    print(f"{name:22s} | changed={changed:4d} | {out}")

print()
print("strict rule matched:", len(strict_changed))
print("balanced rule matched:", len(balanced_changed))

LoRA OFF 후보:
BASEOFF_BLEND10        | changed=   6 | /content/drive/MyDrive/ssafy-16-1-ai-8-28/submission/submission_qwen35_27b_LASTMILE_BASEOFF_BLEND10.csv
BASEOFF_STRICT         | changed=   7 | /content/drive/MyDrive/ssafy-16-1-ai-8-28/submission/submission_qwen35_27b_LASTMILE_BASEOFF_STRICT.csv
BASEOFF_BALANCED       | changed=  10 | /content/drive/MyDrive/ssafy-16-1-ai-8-28/submission/submission_qwen35_27b_LASTMILE_BASEOFF_BALANCED.csv

strict rule matched: 7
balanced rule matched: 10


# 10. Multi-crop 함수

전체 이미지에서 작게 보이는 글자/물체를 확대하기 위해
원본 이미지를 서로 겹치는 5개 crop으로 다시 봅니다.

- center
- top-left
- top-right
- bottom-left
- bottom-right

각 crop은 원본 너비/높이의 약 72%를 사용합니다.
그리고 각 crop마다 선택지 TTA4를 수행합니다.

즉 1문제당 20개의 **fine-tuned LoRA ON** 증거가 추가됩니다.

In [ ]:
CROP_MODES = [
    "center",
    "top_left",
    "top_right",
    "bottom_left",
    "bottom_right",
]

def make_crop(image, mode, fraction=CROP_FRACTION):
    w, h = image.size

    cw = max(2, int(round(w * fraction)))
    ch = max(2, int(round(h * fraction)))

    cw = min(cw, w)
    ch = min(ch, h)

    if mode == "center":
        left = (w - cw) // 2
        top = (h - ch) // 2

    elif mode == "top_left":
        left = 0
        top = 0

    elif mode == "top_right":
        left = w - cw
        top = 0

    elif mode == "bottom_left":
        left = 0
        top = h - ch

    elif mode == "bottom_right":
        left = w - cw
        top = h - ch

    else:
        raise ValueError(mode)

    return image.crop(
        (
            left,
            top,
            left + cw,
            top + ch,
        )
    )

class CropEvalDataset(Dataset):
    def __init__(self, df, perm, crop_mode):
        self.df = df.reset_index(drop=True)
        self.perm = tuple(perm)
        self.crop_mode = crop_mode

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        image = Image.open(
            resolve_test_image_path(row["path"])
        ).convert("RGB")

        crop = make_crop(
            image,
            self.crop_mode,
        )

        orig_options = [str(row[c]) for c in CHOICES]
        shown_options = [orig_options[j] for j in self.perm]

        return {
            "messages": make_messages(
                crop,
                str(row["question"]),
                shown_options,
            ),
            "image": crop,
            "id": str(row["id"]),
        }

@torch.inference_mode()
def predict_crop_original_probs(
    model,
    df,
    perm,
    crop_mode,
    desc,
):
    device = get_model_device(model)

    ds = CropEvalDataset(
        df,
        perm=perm,
        crop_mode=crop_mode,
    )

    dl = DataLoader(
        ds,
        batch_size=INFER_BATCH,
        shuffle=False,
        collate_fn=EvalSubsetCollator(processor),
        num_workers=NUM_WORKERS,
        pin_memory=True,
    )

    choice_ids = CHOICE_TOKEN_IDS.to(device)

    all_original_probs = []
    ids_all = []

    model.eval()

    for enc, ids in tqdm(dl, desc=desc):
        enc = {
            k: (
                v.to(device, non_blocking=True)
                if torch.is_tensor(v)
                else v
            )
            for k, v in enc.items()
        }

        with torch.autocast(
            "cuda",
            dtype=DTYPE,
        ):
            out = model(
                **enc,
                use_cache=False,
            )

        logits_device = out.logits.device

        idx = last_nonpad_index(
            enc["attention_mask"]
        ).to(logits_device)

        bidx = torch.arange(
            out.logits.shape[0],
            device=logits_device,
        )

        next_logits = (
            out.logits[bidx, idx]
            [:, choice_ids.to(logits_device)]
        )

        shown_probs = torch.softmax(
            next_logits.float(),
            dim=-1,
        ).cpu().numpy()

        original_probs = np.zeros_like(shown_probs)

        for shown_j, orig_j in enumerate(perm):
            original_probs[:, orig_j] = shown_probs[:, shown_j]

        all_original_probs.append(original_probs)
        ids_all.extend(ids)

        del out, next_logits, shown_probs, original_probs, enc

    return (
        np.concatenate(all_original_probs, axis=0),
        ids_all,
    )

def cached_crop_predict(
    model,
    df,
    perm,
    crop_mode,
    cache_file,
    desc,
):
    cache_file = Path(cache_file)
    expected_ids = df["id"].astype(str).tolist()

    if cache_file.exists():
        z = np.load(cache_file, allow_pickle=True)

        probs = z["original_probs"]
        ids = [str(x) for x in z["ids"].tolist()]

        assert ids == expected_ids

        print("[cache]", cache_file.name)
        return probs

    probs, ids = predict_crop_original_probs(
        model,
        df,
        perm=perm,
        crop_mode=crop_mode,
        desc=desc,
    )

    assert ids == expected_ids

    cache_file.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    np.savez_compressed(
        cache_file,
        original_probs=probs,
        ids=np.array(ids, dtype=object),
        perm=np.array(perm, dtype=np.int64),
        crop_mode=np.array(crop_mode),
    )

    print("saved:", cache_file)

    return probs

## 11. Multi-crop 대상 200개 선택 + 추론

우선순위:
1. LoRA OFF와 현재 best가 서로 다른 문제
2. 현재 best margin이 낮은 문제
3. Adaptive TTA24 uncertainty가 높은 문제

따라서 단순히 가장 애매한 200개보다 **두 모델이 실제로 충돌하는 문제**에 계산량을 더 집중합니다.

In [ ]:
# baseoff subset 내에서 multi-crop 우선순위 계산
baseoff_current = current_best_probs[baseoff_global_indices]
baseoff_current_pred = baseoff_current.argmax(axis=1)
baseoff_model_pred = baseoff_mean.argmax(axis=1)

baseoff_current_sorted = np.sort(baseoff_current, axis=1)
baseoff_current_margin = (
    baseoff_current_sorted[:, -1]
    - baseoff_current_sorted[:, -2]
)

# disagreement에 가장 큰 가중치
multi_priority = (
    1.20 * (baseoff_current_pred != baseoff_model_pred).astype(np.float64)
    +
    0.80 * (1.0 - baseoff_current_margin)
    +
    0.50 * baseoff_unc["score"]
)

multi_local = np.argsort(
    -multi_priority
)[:min(MULTICROP_COUNT, len(baseoff_df))]

multicrop_global_indices = baseoff_global_indices[multi_local]
multicrop_global_indices = np.sort(multicrop_global_indices)

multicrop_df = (
    test_df
    .iloc[multicrop_global_indices]
    .copy()
    .reset_index(drop=True)
)

multicrop_ids = multicrop_df["id"].astype(str).tolist()

multicrop_hash = hashlib.sha1(
    "\n".join(multicrop_ids).encode("utf-8")
).hexdigest()[:10]

MULTICROP_CACHE_DIR = (
    LASTMILE_DIR
    /
    (
        f"multicrop_v1024_n{len(multicrop_df)}_"
        f"{multicrop_hash}_f{int(round(CROP_FRACTION*100))}_"
        f"tta{MULTICROP_TTA_PERMS}"
    )
)

selected_crop_perms = BASE_PERMS[:MULTICROP_TTA_PERMS]

crop_runs = []

total_runs = len(CROP_MODES) * len(selected_crop_perms)
run_no = 0

for crop_mode in CROP_MODES:
    for pi, perm in enumerate(selected_crop_perms, start=1):
        run_no += 1

        perm_key = "".join(map(str, perm))

        f = (
            MULTICROP_CACHE_DIR
            /
            f"{crop_mode}_perm{pi}_{perm_key}.npz"
        )

        probs = cached_crop_predict(
            final_model,
            multicrop_df,
            perm=perm,
            crop_mode=crop_mode,
            cache_file=f,
            desc=(
                f"MULTICROP {run_no}/{total_runs} "
                f"{crop_mode} p{pi}"
            ),
        )

        crop_runs.append(probs)

crop_runs = np.stack(crop_runs, axis=0)

crop_mean = crop_runs.mean(axis=0)
crop_unc = uncertainty_metrics(crop_runs)

current_crop_subset = current_best_probs[multicrop_global_indices]

print()
print("crop runs:", crop_runs.shape)
print(
    "current best와 crop mean 답이 다른 수:",
    int(
        (
            current_crop_subset.argmax(axis=1)
            != crop_mean.argmax(axis=1)
        ).sum()
    ),
)
print(
    "crop evidence >= 80% vote agreement:",
    int((crop_unc["vote_agreement"] >= 0.80).sum()),
    "/",
    len(multicrop_df),
)

MULTICROP 1/20 center p1:   0%|          | 0/200 [00:00<?, ?it/s]

saved: /content/drive/MyDrive/ssafy-16-1-ai-8-28/lastmile_qwen35_27b/multicrop_v1024_n200_9946b946ea_f72_tta4/center_perm1_0123.npz


MULTICROP 2/20 center p2:   0%|          | 0/200 [00:00<?, ?it/s]

saved: /content/drive/MyDrive/ssafy-16-1-ai-8-28/lastmile_qwen35_27b/multicrop_v1024_n200_9946b946ea_f72_tta4/center_perm2_1302.npz


MULTICROP 3/20 center p3:   0%|          | 0/200 [00:00<?, ?it/s]

...

saved: /content/drive/MyDrive/ssafy-16-1-ai-8-28/lastmile_qwen35_27b/multicrop_v1024_n200_9946b946ea_f72_tta4/bottom_right_perm2_1302.npz


MULTICROP 19/20 bottom_right p3:   0%|          | 0/200 [00:00<?, ?it/s]

saved: /content/drive/MyDrive/ssafy-16-1-ai-8-28/lastmile_qwen35_27b/multicrop_v1024_n200_9946b946ea_f72_tta4/bottom_right_perm3_2031.npz


MULTICROP 20/20 bottom_right p4:   0%|          | 0/200 [00:00<?, ?it/s]

saved: /content/drive/MyDrive/ssafy-16-1-ai-8-28/lastmile_qwen35_27b/multicrop_v1024_n200_9946b946ea_f72_tta4/bottom_right_perm4_3210.npz

crop runs: (20, 200, 4)
current best와 crop mean 답이 다른 수: 79
crop evidence >= 80% vote agreement: 72 / 200


## 12. Multi-crop / 3-way consensus 후보 생성



여기서 가장 중요한 후보는 `TRIPLE_STRICT`입니다.

현재 best와 다른 답으로 바꾸려면 대체로:
- 현재 best의 margin이 낮고
- LoRA OFF base model과
- multi-crop fine-tuned model이
- **같은 새로운 답에 동의**
해야 합니다.

즉 한 가지 기법의 흔들림만으로 답을 바꾸지 않습니다.

In [ ]:
# global -> baseoff local
global_to_baseoff_local = {
    int(g): i
    for i, g in enumerate(baseoff_global_indices)
}

# global -> multicrop local
global_to_multicrop_local = {
    int(g): i
    for i, g in enumerate(multicrop_global_indices)
}

FINAL_OUTPUTS = []

# ------------------------------------------------------------
# A. crop soft blend 15%, 25%
# ------------------------------------------------------------
for w in [0.15, 0.25]:
    p = current_best_probs.copy()

    p[multicrop_global_indices] = (
        (1.0 - w)
        * current_best_probs[multicrop_global_indices]
        +
        w
        * crop_mean
    )

    out, _ = save_submission_from_probs(
        p,
        f"submission_qwen35_27b_LASTMILE_MULTICROP_W{int(w*100)}.csv",
    )

    FINAL_OUTPUTS.append(
        (
            f"MULTICROP_W{int(w*100)}",
            out,
            int((p.argmax(axis=1) != current_best_pred).sum()),
        )
    )

# ------------------------------------------------------------
# B. triple strict
# ------------------------------------------------------------
p_triple = current_best_probs.copy()
triple_rows = []

for global_i in multicrop_global_indices:
    global_i = int(global_i)

    bo_i = global_to_baseoff_local[global_i]
    mc_i = global_to_multicrop_local[global_i]

    cur_p = current_best_probs[global_i]
    cur_pred = int(cur_p.argmax())

    bo_p = baseoff_mean[bo_i]
    bo_pred = int(bo_p.argmax())

    mc_p = crop_mean[mc_i]
    mc_pred = int(mc_p.argmax())

    # 두 독립 증거가 같은 새 답이어야 함
    if not (
        bo_pred == mc_pred
        and bo_pred != cur_pred
    ):
        continue

    cur_gap = float(
        np.sort(cur_p)[-1] - np.sort(cur_p)[-2]
    )

    bo_conf = float(bo_p[bo_pred])
    bo_agree = float(baseoff_unc["vote_agreement"][bo_i])

    mc_conf = float(mc_p[mc_pred])
    mc_agree = float(crop_unc["vote_agreement"][mc_i])

    # 현재 FT TTA24의 해당 global index
    hard_local = global_to_hard_local[global_i]
    ft24_p = hard_arith[hard_local]

    support_gap = float(
        ft24_p[cur_pred] - ft24_p[bo_pred]
    )

    ok = (
        cur_gap < 0.12
        and bo_conf >= 0.58
        and bo_agree >= 0.75
        and mc_conf >= 0.55
        and mc_agree >= 0.70
        and support_gap <= 0.12
    )

    if ok:
        # current / base / crop 세 증거를 결합
        p_triple[global_i] = (
            0.30 * cur_p
            +
            0.35 * bo_p
            +
            0.35 * mc_p
        )

        triple_rows.append({
            "global_index": global_i,
            "id": str(test_df.iloc[global_i]["id"]),
            "current": CHOICES[cur_pred],
            "new_consensus": CHOICES[bo_pred],
            "current_margin": cur_gap,
            "base_conf": bo_conf,
            "base_vote_agree": bo_agree,
            "crop_conf": mc_conf,
            "crop_vote_agree": mc_agree,
            "ft24_support_gap": support_gap,
        })

out_triple, _ = save_submission_from_probs(
    p_triple,
    "submission_qwen35_27b_LASTMILE_TRIPLE_STRICT.csv",
)

FINAL_OUTPUTS.append(
    (
        "TRIPLE_STRICT",
        out_triple,
        int((p_triple.argmax(axis=1) != current_best_pred).sum()),
    )
)

triple_df = pd.DataFrame(triple_rows)

TRIPLE_DIAG = LASTMILE_DIR / "triple_strict_changes.csv"
triple_df.to_csv(TRIPLE_DIAG, index=False)

# ------------------------------------------------------------
# C. triple + 조금 더 보수적인 crop/base 평균 후보
# ------------------------------------------------------------
p_consensus_soft = current_best_probs.copy()

for global_i in multicrop_global_indices:
    global_i = int(global_i)

    bo_i = global_to_baseoff_local[global_i]
    mc_i = global_to_multicrop_local[global_i]

    cur_p = current_best_probs[global_i]
    cur_pred = int(cur_p.argmax())

    bo_p = baseoff_mean[bo_i]
    mc_p = crop_mean[mc_i]

    bo_pred = int(bo_p.argmax())
    mc_pred = int(mc_p.argmax())

    if (
        bo_pred == mc_pred
        and bo_pred != cur_pred
        and current_margin[global_i] < 0.15
        and baseoff_unc["vote_agreement"][bo_i] >= 0.75
        and crop_unc["vote_agreement"][mc_i] >= 0.65
    ):
        p_consensus_soft[global_i] = (
            0.50 * cur_p
            +
            0.25 * bo_p
            +
            0.25 * mc_p
        )

out_consensus_soft, _ = save_submission_from_probs(
    p_consensus_soft,
    "submission_qwen35_27b_LASTMILE_TRIPLE_SOFT.csv",
)

FINAL_OUTPUTS.append(
    (
        "TRIPLE_SOFT",
        out_consensus_soft,
        int((p_consensus_soft.argmax(axis=1) != current_best_pred).sum()),
    )
)

print("최종 후보:")
for name, out, changed in FINAL_OUTPUTS:
    print(f"{name:20s} | changed={changed:4d} | {out}")

print()
print("triple diagnostics:", TRIPLE_DIAG)
print("triple selected rows:", len(triple_df))

최종 후보:
MULTICROP_W15        | changed=  12 | /content/drive/MyDrive/ssafy-16-1-ai-8-28/submission/submission_qwen35_27b_LASTMILE_MULTICROP_W15.csv
MULTICROP_W25        | changed=  18 | /content/drive/MyDrive/ssafy-16-1-ai-8-28/submission/submission_qwen35_27b_LASTMILE_MULTICROP_W25.csv
TRIPLE_STRICT        | changed=   4 | /content/drive/MyDrive/ssafy-16-1-ai-8-28/submission/submission_qwen35_27b_LASTMILE_TRIPLE_STRICT.csv
TRIPLE_SOFT          | changed=  12 | /content/drive/MyDrive/ssafy-16-1-ai-8-28/submission/submission_qwen35_27b_LASTMILE_TRIPLE_SOFT.csv

triple diagnostics: /content/drive/MyDrive/ssafy-16-1-ai-8-28/lastmile_qwen35_27b/triple_strict_changes.csv
triple selected rows: 4


# 13. 최종 제출 후보 정리

추천 순서는 다음과 같습니다.

1. **TRIPLE_STRICT**
   - 현재 best를 가장 보수적으로 수정
   - LoRA OFF base와 multi-crop FT가 같은 새 답에 동의할 때 중심적으로 수정

2. **BASEOFF_STRICT**
   - crop 없이 원본 Qwen3.5-27B의 강한 의견만 사용

3. **MULTICROP_W15**
   - 작은 글씨/물체에 대한 약한 보정

4. **TRIPLE_SOFT**
   - strict보다 답이 조금 더 많이 변할 수 있음

5. **BASEOFF_BLEND10**

무료 후보:
- HiRes W45
- HiRes W50

중요:
- 기존 `0.95072` CSV는 삭제하거나 덮어쓰지 않습니다.
- 새로운 후보는 별도 파일명으로 저장됩니다.
- 캐글 점수를 특정 개별 row의 정답을 역추적하는 용도로 사용하지 마세요.

In [ ]:
print("=" * 90)
print("RECOMMENDED SUBMISSIONS")
print("=" * 90)

all_candidates = (
    [("FREE_W45", FREE_OUTPUTS[0][1], FREE_OUTPUTS[0][2])]
    + [("FREE_W50", FREE_OUTPUTS[1][1], FREE_OUTPUTS[1][2])]
    + BASEOFF_OUTPUTS
    + FINAL_OUTPUTS
)

recommended_order = [
    "TRIPLE_STRICT",
    "BASEOFF_STRICT",
    "MULTICROP_W15",
    "TRIPLE_SOFT",
    "BASEOFF_BLEND10",
    "FREE_W45",
    "FREE_W50",
    "MULTICROP_W25",
    "BASEOFF_BALANCED",
]

lookup = {
    name: (out, changed)
    for name, out, changed in all_candidates
}

for rank, name in enumerate(recommended_order, start=1):
    if name not in lookup:
        continue

    out, changed = lookup[name]

    print(
        f"{rank:2d}. {name:20s} "
        f"| changed={changed:4d} "
        f"| {out}"
    )

print()
print("현재 best는 그대로 보존하세요: 0.95072")
print("=" * 90)

RECOMMENDED SUBMISSIONS
 1. TRIPLE_STRICT        | changed=   4 | /content/drive/MyDrive/ssafy-16-1-ai-8-28/submission/submission_qwen35_27b_LASTMILE_TRIPLE_STRICT.csv
 2. BASEOFF_STRICT       | changed=   7 | /content/drive/MyDrive/ssafy-16-1-ai-8-28/submission/submission_qwen35_27b_LASTMILE_BASEOFF_STRICT.csv
 3. MULTICROP_W15        | changed=  12 | /content/drive/MyDrive/ssafy-16-1-ai-8-28/submission/submission_qwen35_27b_LASTMILE_MULTICROP_W15.csv
 4. TRIPLE_SOFT          | changed=  12 | /content/drive/MyDrive/ssafy-16-1-ai-8-28/submission/submission_qwen35_27b_LASTMILE_TRIPLE_SOFT.csv
 5. BASEOFF_BLEND10      | changed=   6 | /content/drive/MyDrive/ssafy-16-1-ai-8-28/submission/submission_qwen35_27b_LASTMILE_BASEOFF_BLEND10.csv
 6. FREE_W45             | changed=   2 | /content/drive/MyDrive/ssafy-16-1-ai-8-28/submission/submission_qwen35_27b_ADAPTIVE_TTA24_H1000_HI1536_N400_W45.csv
 7. FREE_W50             | changed=   2 | /content/drive/MyDrive/ssafy-16-1-ai-8-28/submission/su

## 런타임이 끊기면

LoRA OFF와 multi-crop 모두 각 permutation/crop 완료 시 Drive에 cache됩니다.

다시 실행하면:

```text
[cache] baseoff_...
[cache] center_perm...
```

처럼 완료된 계산은 재사용합니다.

시간이 빠듯해지면 상단에서:

```python
MULTICROP_COUNT = 120
```

으로 줄일 수 있습니다.

LoRA OFF 500개 TTA4는 가능한 한 유지하는 것을 추천합니다.